In [ ]:
!pip install transformers datasets

In [ ]:
import pandas as pd
from datasets import Dataset
from transformers import RobertaTokenizerFast

In [ ]:
file_path = '/content/Qualitative_coding_emotional_needs_TLR_LH (1).xlsx'
data = pd.read_excel(file_path)

In [ ]:
data.head()

In [ ]:
data = data[['question_content/Original Post', 'Emotional support need']].dropna()
dataset = Dataset.from_pandas(data)

In [ ]:
tokenizer = RobertaTokenizerFast.from_pretrained('roberta-base')
def tokenize_function(examples):
    return tokenizer(examples['question_content/Original Post'], padding="max_length", truncation=True, max_length=256)

tokenized_datasets = dataset.map(tokenize_function, batched=True)
tokenized_datasets = tokenized_datasets.rename_column("Emotional support need", "labels")
tokenized_datasets.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

In [ ]:
from sklearn.model_selection import train_test_split

train_test_val_dataset = tokenized_datasets.train_test_split(test_size=0.2)
test_dataset = train_test_val_dataset['test']
train_val_dataset = train_test_val_dataset['train'].train_test_split(test_size=0.1)
train_dataset = train_val_dataset['train']
val_dataset = train_val_dataset['test']

In [ ]:
from sklearn.metrics import precision_recall_fscore_support, accuracy_score

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)
    return {'accuracy': acc, 'f1': f1, 'precision': precision, 'recall': recall}

In [ ]:
from transformers import RobertaForSequenceClassification, TrainingArguments, Trainer, AutoConfig

config = AutoConfig.from_pretrained('roberta-base', num_labels=2)
model = RobertaForSequenceClassification.from_pretrained('roberta-base', config=config)

In [ ]:
pip install accelerate -U

In [ ]:
training_args = TrainingArguments(
    output_dir='./content/results',
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    logging_dir='./content/logs',
    logging_steps=10,
    report_to="none"
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

evaluation_results = trainer.evaluate()
print(evaluation_results)